# Lab 04: Amazon DynamoDB (boto3)

Follows the lab guide with API calls.

## Setup

Settings come from `.env`. Checking target to know wher ecalls go.

In [ ]:
import os

import boto3
from boto3.dynamodb.conditions import Key, Attr
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(), override=True)
print("Target:", os.environ.get("AWS_ENDPOINT_URL", "real AWS"))

TABLE_NAME = "Music"
client = boto3.client('dynamodb')
dynamodb = boto3.resource('dynamodb')


Removing any leftover table from previous run:

In [ ]:
try:
    client.delete_table(TableName=TABLE_NAME)
    client.get_waiter('table_not_exists').wait(TableName=TABLE_NAME)
except client.exceptions.ResourceNotFoundException:
    pass

## Task 1: Create a New Table
`Artist` is the **partition key**, `Song` is the **sort key**

In [ ]:
client.create_table(
    TableName = TABLE_NAME,
    KeySchema = [
        {'AttributeName': 'Artist', 'KeyType': 'HASH'},
        {'AttributeName': 'Song',   'KeyType': 'RANGE'},
    ],
    AttributeDefinitions = [
        {'AttributeName': 'Artist', 'AttributeType': 'S'},
        {'AttributeName': 'Song',   'AttributeType': 'S'},
    ],
    BillingMode = 'PAY_PER_REQUEST',
)

client.get_waiter('table_exists').wait(TableName=TABLE_NAME)
print(client.describe_table(TableName=TABLE_NAME)['Table']['TableStatus'])

## Task 2: Add Data
Items may carry different attributes

In [ ]:
table = dynamodb.Table(TABLE_NAME)

songs = [
    {'Artist': 'Pink Floyd', 'Song': 'Money',
        'Album': 'The Dark Side of the Moon', 'Year': 1973},
    {'Artist': 'John Lennon', 'Song': 'Imagine',
        'Album': 'Imagine', 'Year': 1971, 'Genre': 'Soft rock'},
    {'Artist': 'Psy', 'Song': 'Gangnam Style',
        'Album': 'Psy 6 (Six Rules), Part 1',
        'Year': 2011, 'LengthSeconds': 219},
]

for song in songs:
    table.put_item(Item=song)

assert table.scan(TableName=TABLE_NAME)['Count'] == len(songs)

## Task 3: Modify an Existing Item

In [ ]:
psy = {'Artist': 'Psy', 'Song': 'Gangnam Style'}

table.update_item(
    Key=psy,
    UpdateExpression = 'SET #year = :year',
    ExpressionAttributeNames = {'#year': 'Year'},
    ExpressionAttributeValues = {':year': 2012},
)

assert table.get_item(Key=psy)['Item']['Year'] == 2012

## Task 4: Query and Scan
A query reads only the items under one key. \
A scan reads the whole table and filters afterwards, so `ScannedCount` grows with the table while `Count` stays small.

In [ ]:
query = table.query(
    KeyConditionExpression=Key('Artist').eq('Psy')
            & Key('Song').eq('Gangnam Style')
)
assert query['Count'] == 1

scan = table.scan(
    FilterExpression=Key('Year').eq(1971)
)
print(f"Returned {scan['Count']}, read {scan['ScannedCount']}")
(item,) = scan['Items'] # raises if more than one
assert item['Song'] == 'Imagine'

## Task 5: Delete the Table

In [ ]:
client.delete_table(TableName=TABLE_NAME)
client.get_waiter('table_not_exists').wait(TableName=TABLE_NAME)
assert TABLE_NAME not in client.list_tables()["TableNames"]